# LatentSR - AdaGN on VAE-SR (Kaggle)

Train the **spatial-FiLM conditioner** (`condition_type: adagn`) with the **frozen Q2 VAE-SR**. Same 50-epoch recipe as concat Q2. Do **not** resume a concat / Phase-8 checkpoint.

### Before Run All
1. Settings → Accelerator: **GPU**
2. Settings → Internet: **ON**
3. Add secret `HF_TOKEN` (write access to `HusseinHamouda/LatentSR-checkpoints`)
4. AdaGN must already be on GitHub (`main`). This notebook clones the repo; it does not upload local uncommitted code.

HF writes go to `latent_sr_adagn_q2/` and will not overwrite concat Q2.

If this session dies, **Run All** again: CelebA and VAE download skip if present, and training resumes from local `latest.pt` or HF `latent_sr_adagn_q2/latest.pt`.


## 1. Hugging Face login


In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami, HfApi

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)

print("HF account:", whoami()["name"])
api = HfApi()
api.repo_info(repo_id="HusseinHamouda/LatentSR-checkpoints", repo_type="model")
print("Repository access: OK")

HF account: HusseinHamouda
Repository access: OK


## 2. Clone LatentSR + generative-models


In [2]:
from pathlib import Path
import subprocess
import sys

def run(cmd, cwd=None):
    print("+", " ".join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.check_call(cmd, cwd=cwd)


root = Path("/kaggle/working")
latentsr = root / "LatentSR"
gm = root / "generative-models"

if latentsr.exists():
    run(["git", "-C", str(latentsr), "pull"])
else:
    run(["git", "clone", "https://github.com/HusseinHanafy207/LatentSR.git", str(latentsr)])

if gm.exists():
    run(["git", "-C", str(gm), "pull"])
else:
    run(["git", "clone", "https://github.com/HusseinHanafy207/generative-models.git", str(gm)])

py = sys.executable
run([py, "-m", "pip", "install", "-q", "-e", str(gm)])
run([py, "-m", "pip", "install", "-q", "-e", str(latentsr)])
run([py, "-m", "pip", "install", "-q", "gdown", "huggingface_hub"])
print("install done")


+ git clone https://github.com/HusseinHanafy207/LatentSR.git /kaggle/working/LatentSR


Cloning into '/kaggle/working/LatentSR'...


+ git clone https://github.com/HusseinHanafy207/generative-models.git /kaggle/working/generative-models


Cloning into '/kaggle/working/generative-models'...


+ /usr/bin/python3 -m pip install -q -e /kaggle/working/generative-models
+ /usr/bin/python3 -m pip install -q -e /kaggle/working/LatentSR
+ /usr/bin/python3 -m pip install -q gdown huggingface_hub
install done


## 3. CelebA (Drive copies)

Torchvision cannot download CelebA on Kaggle. These are the same Drive file IDs from the previous phases. Re-running skips files that already exist.


In [3]:
from pathlib import Path
import subprocess

celeba = Path("/kaggle/working/data/raw/celeba")
celeba.mkdir(parents=True, exist_ok=True)
img_dir = celeba / "img_align_celeba"

# Replace only if a Drive link rotates.
files = {
    "img_align_celeba.zip": "1lVqCbFGvz_zEwFwGXZDE57fZgzyE54lX",
    "list_attr_celeba.txt": "1--ygZyBF_NVgZV0ghGdEKKw-RDT_CUsf",
    "list_eval_partition.txt": "1sfBkt6LULyQYBfEQenI-X2LcCaoiZkwk",
    "identity_CelebA.txt": "1_c8h_buw8wnTW_cfwk2KfpY-i6dUaKPH",
    "list_bbox_celeba.txt": "1KShnIpocgfBlBJ-2IUPAfEDJyOpvKF_c",
    "list_landmarks_align_celeba.txt": "1K_ycTaSHqhyiIfBohSPMDUXKdp0Dezpq",
}

def jpg_count(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.iterdir() if p.suffix.lower() == ".jpg")


n_img = jpg_count(img_dir)
if n_img >= 200000:
    print(f"CelebA already extracted ({n_img} jpg). Skip download.")
else:
    for name, file_id in files.items():
        dest = celeba / name
        if dest.exists() and dest.stat().st_size > 0:
            print("exists", dest.name)
            continue
        url = f"https://drive.google.com/uc?id={file_id}"
        subprocess.check_call(["gdown", url, "-O", str(dest)])
    zip_path = celeba / "img_align_celeba.zip"
    print("unzipping", zip_path)
    subprocess.check_call(["unzip", "-q", "-o", str(zip_path), "-d", str(celeba)])
    n_img = jpg_count(img_dir)
    print("jpg count:", n_img)
    if n_img < 200000:
        raise RuntimeError(f"expected ~202599 faces, got {n_img}")


Downloading...
From (original): https://drive.google.com/uc?id=1lVqCbFGvz_zEwFwGXZDE57fZgzyE54lX
From (redirected): https://drive.google.com/uc?id=1lVqCbFGvz_zEwFwGXZDE57fZgzyE54lX&confirm=t&uuid=91114e3e-f78f-4c84-b514-5d7466b784c8
To: /kaggle/working/data/raw/celeba/img_align_celeba.zip
100%|██████████| 1.44G/1.44G [00:15<00:00, 93.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1--ygZyBF_NVgZV0ghGdEKKw-RDT_CUsf
To: /kaggle/working/data/raw/celeba/list_attr_celeba.txt
100%|██████████| 26.7M/26.7M [00:00<00:00, 82.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1sfBkt6LULyQYBfEQenI-X2LcCaoiZkwk
To: /kaggle/working/data/raw/celeba/list_eval_partition.txt
100%|██████████| 2.84M/2.84M [00:00<00:00, 198MB/s]
Downloading...
From: https://drive.google.com/uc?id=1_c8h_buw8wnTW_cfwk2KfpY-i6dUaKPH
To: /kaggle/working/data/raw/celeba/identity_CelebA.txt
100%|██████████| 3.42M/3.42M [00:00<00:00, 175MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KShnIpocgfBlBJ-2

unzipping /kaggle/working/data/raw/celeba/img_align_celeba.zip
jpg count: 202599


## 4. Frozen VAE-SR (HF) - not VAE-1, not a DDPM ckpt


In [4]:
from pathlib import Path
import shutil
import torch
from huggingface_hub import hf_hub_download

vae_dest = Path("/kaggle/working/outputs/vae_sr/checkpoints/latest.pt")
vae_dest.parent.mkdir(parents=True, exist_ok=True)

src = Path(
    hf_hub_download(
        repo_id="HusseinHamouda/LatentSR-checkpoints",
        filename="vae_sr/latest.pt",
        local_dir="/kaggle/working/hf_ckpt",
    )
)
if src.resolve() != vae_dest.resolve():
    shutil.copy2(src, vae_dest)

ckpt = torch.load(vae_dest, map_location="cpu", weights_only=False)
print("VAE-SR path:", vae_dest)
print("VAE-SR epoch:", ckpt.get("epoch"))
print("keys:", sorted(ckpt.keys())[:20])
if "model_state_dict" not in ckpt:
    raise RuntimeError("VAE checkpoint missing model_state_dict")


vae_sr/latest.pt:   0%|          | 0.00/83.6M [00:00<?, ?B/s]

VAE-SR path: /kaggle/working/outputs/vae_sr/checkpoints/latest.pt
VAE-SR epoch: 20
keys: ['arch', 'config', 'epoch', 'metrics', 'model_state_dict', 'optimizer_state_dict']


## 5. Confirm AdaGN config (do not concat)


In [5]:
from pathlib import Path
import yaml

cfg_path = Path("/kaggle/working/LatentSR/configs/latent_sr_adagn_q2.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
print("config:", cfg_path)
print("condition_type:", cfg.get("condition_type"))
print("vae_checkpoint:", cfg.get("vae_checkpoint"))
print("checkpoint_dir:", cfg.get("checkpoint_dir"))
print("hf_checkpoint_subdir:", cfg.get("hf_checkpoint_subdir"))
if cfg.get("condition_type") not in {"adagn", "film"}:
    raise RuntimeError(f"expected adagn, got {cfg.get('condition_type')!r}")

out = Path("/kaggle/working/outputs/latent_sr_adagn_q2")
for sub in ("checkpoints", "samples", "logs"):
    (out / sub).mkdir(parents=True, exist_ok=True)
print("output dirs ready")


config: /kaggle/working/LatentSR/configs/latent_sr_adagn_q2.yaml
condition_type: adagn
vae_checkpoint: /kaggle/working/outputs/vae_sr/checkpoints/latest.pt
checkpoint_dir: /kaggle/working/outputs/latent_sr_adagn_q2/checkpoints
hf_checkpoint_subdir: latent_sr_adagn_q2
output dirs ready


## 6. Train 50 epochs

Set `EPOCHS = 1` for a smoke test, then re-run this cell with `EPOCHS = 50`. If `latest.pt` exists locally or on HF under `latent_sr_adagn_q2/`, this cell resumes. It **refuses** a concat checkpoint.


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
from huggingface_hub import hf_hub_download

EPOCHS = 50  # 1 = smoke test only
CONFIG = "/kaggle/working/LatentSR/configs/latent_sr_adagn_q2.yaml"
VAE = "/kaggle/working/outputs/vae_sr/checkpoints/latest.pt"
LOCAL = Path("/kaggle/working/outputs/latent_sr_adagn_q2/checkpoints/latest.pt")
REPO = "HusseinHamouda/LatentSR-checkpoints"


def condition_type_of(path: Path) -> str:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    ctype = (ckpt.get("config") or {}).get("condition_type", "concat")
    print(f"{path}: epoch={ckpt.get('epoch')} condition_type={ctype}")
    if ctype not in {"adagn", "film"}:
        raise RuntimeError(
            f"Refusing to resume {path} (condition_type={ctype}). "
            "AdaGN cannot load a concat checkpoint."
        )
    return ctype


resume = None
if LOCAL.exists():
    condition_type_of(LOCAL)
    resume = str(LOCAL)
else:
    try:
        hf_path = Path(
            hf_hub_download(
                repo_id=REPO,
                filename="latent_sr_adagn_q2/latest.pt",
                local_dir="/kaggle/working/hf_ckpt",
            )
        )
        condition_type_of(hf_path)
        resume = str(hf_path)
    except Exception as exc:
        print("No AdaGN checkpoint yet; training from scratch.")
        print(" ", exc)

cmd = [
    sys.executable,
    "scripts/train_sr.py",
    "--config",
    CONFIG,
    "--vae-checkpoint",
    VAE,
    "--epochs",
    str(EPOCHS),
    "--device",
    "cuda",
    "--no-download",
]
if resume:
    cmd += ["--resume", resume]

print("+", " ".join(cmd))
subprocess.check_call(cmd, cwd="/kaggle/working/LatentSR")


No AdaGN checkpoint yet; training from scratch.
  404 Client Error. (Request ID: Root=1-6a84fb40-4c82801e09c796d51033b645;34d444d4-5b7b-4f19-8da1-9be422ca2042)

Entry Not Found for url: https://huggingface.co/HusseinHamouda/LatentSR-checkpoints/resolve/main/latent_sr_adagn_q2/latest.pt.
+ /usr/bin/python3 scripts/train_sr.py --config /kaggle/working/LatentSR/configs/latent_sr_adagn_q2.yaml --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt --epochs 50 --device cuda --no-download
condition_type=adagn  params=16.73M
Device: cuda  |  AMP: True
Train pairs: 162770  |  batches/epoch: 2544  |  batch_size: 64
VAE: /kaggle/working/outputs/vae_sr/checkpoints/latest.pt  |  latent_scale: 1.0  |  condition: adagn
HF backup every epoch → hf://HusseinHamouda/LatentSR-checkpoints/latent_sr_adagn_q2/



Epoch [1/50]
Train Loss:  0.171974
Time:        1662.6 sec
Val Loss:    0.153383
Saved /kaggle/working/outputs/latent_sr_adagn_q2/checkpoints/latest.pt (191.8 MB, epoch 1)
Saved snapshot checkpoint_epoch_001.pt
Generating qualitative SR grid (may take a few minutes)…
Saved comparison grid to /kaggle/working/outputs/latent_sr_adagn_q2/samples/sr_compare_epoch_001.png


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ..._q2/checkpoints/latest.pt:   0%|          |  562kB /  201MB            

Processing Files (0 / 1)      :   0%|          |  562kB /  201MB,  281kB/s  
New Data Upload               :   0%|          |  562kB /  134MB,  281kB/s  

  ..._q2/checkpoints/latest.pt:   0%|          |  562kB /  201MB            

  ..._q2/checkpoints/latest.pt:   0%|          |  562kB /  201MB            

Processing Files (0 / 1)      :   1%|          | 1.12MB /  201MB,  432kB/s  
New Data Upload               :   1%|          | 1.12MB /  134MB,  432kB/s  

Processing Files (0 / 1)      :   1%|          | 1.69MB /  201MB,  602kB/s  
New Data Upload               :   1%|▏         | 1.69MB /  134MB,  602kB/s  

Processing Files (0 / 1)      :   2%|▏         | 4.50MB /  201MB, 1.50MB/s  
New Data Upload               :   3%|▎         | 4.50MB /  134MB, 1.50MB/s  


  HF uploaded latent_sr_adagn_q2/latest.pt


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ..._q2/checkpoints/latest.pt:  88%|████████▊ |  176MB /  201MB            

Processing Files (0 / 1)      :  88%|████████▊ |  176MB /  201MB,   ???B/s  

Processing Files (1 / 1)      : 100%|██████████|  201MB /  201MB,  126MB/s  

  ..._q2/checkpoints/latest.pt: 100%|██████████|  201MB /  201MB            

  ..._q2/checkpoints/latest.pt: 100%|██████████|  201MB /  201MB            

  ..._q2/checkpoints/latest.pt: 100%|██████████|  201MB /  201MB            

  ..._q2/checkpoints/latest.pt: 100%|██████████|  201MB /  201MB            

Processing Files (1 / 1)      : 100%|██████████|  201MB /  201MB, 25.1MB/s  
New Data Upload               : |          |  0.00B /  0.00B,  0.00B/s  
  ..._q2/checkpoints/latest.pt: 100%|██████████|  201MB /  201MB            


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ..._q2/checkpoints/latest.pt:   0%|          | 65.9kB /  201MB            

Processing Files (0 / 1)      :   0%|          | 65.9kB /  201MB,  110kB/s  

  ..._q2/checkpoints/latest.pt:   0%|          | 65.9kB /  201MB            

  ..._q2/checkpoints/latest.pt:   0%|          | 65.9kB /  201MB            

  ..._q2/checkpoints/latest.pt:   0%|          | 65.9kB /  201MB            

  ..._q2/checkpoints/latest.pt:   0%|          | 65.9kB /  201MB            

Processing Files (0 / 1)      :   0%|          |  622kB /  201MB,  388kB/s  
New Data Upload               :   1%|          |  556kB / 67.0MB,  347kB/s  

  ..._q2/checkpoints/latest.pt:   0%|          |  622kB /  201MB            

  ..._q2/checkpoints/latest.pt:   0%|          |  622kB /  201MB            

Processing Files (0 / 1)      :   1%|          | 1.18MB /  201MB,  535kB/s

  HF uploaded latent_sr_adagn_q2/latest.pt


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ..._q2/checkpoints/latest.pt:  92%|█████████▏|  184MB /  201MB            

Processing Files (0 / 1)      :  92%|█████████▏|  184MB /  201MB,   ???B/s  

Processing Files (1 / 1)      : 100%|██████████|  201MB /  201MB, 85.1MB/s  

  ..._q2/checkpoints/latest.pt: 100%|██████████|  201MB /  201MB            

  ..._q2/checkpoints/latest.pt: 100%|██████████|  201MB /  201MB            

  ..._q2/checkpoints/latest.pt: 100%|██████████|  201MB /  201MB            

  ..._q2/checkpoints/latest.pt: 100%|██████████|  201MB /  201MB            

Processing Files (1 / 1)      : 100%|██████████|  201MB /  201MB, 17.1MB/s  
New Data Upload               : |          |  0.00B /  0.00B,  0.00B/s  
  ..._q2/checkpoints/latest.pt: 100%|██████████|  201MB /  201MB            
No files have been modified since last commit. Skipping to prevent empty commit.

  HF uploaded latent_sr_adagn_q2/checkpoint_epoch_024.pt
  HF uploaded latent_sr_adagn_q2/logs/train_metrics.csv
  HF uploaded latent_sr_adagn_q2/logs/val_metrics.csv
HF backup complete for epoch 24 → hf://HusseinHamouda/LatentSR-checkpoints/latent_sr_adagn_q2/ (4 files)



train:  99%|█████████▊| 2512/2544 [27:25<00:20,  1.53it/s, loss=0.0969]

In [ ]:
%cd /kaggle/working/LatentSR

from pathlib import Path
from huggingface_hub import hf_hub_download
import torch

ckpt_path = hf_hub_download(
    repo_id="HusseinHamouda/LatentSR-checkpoints",
    filename="latent_sr_adagn_q2/latest.pt",
    local_dir="/kaggle/working/hf_ckpt",
)
print("resume from:", ckpt_path)

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print("epoch:", ckpt.get("epoch"))
print("condition_type:", (ckpt.get("config") or {}).get("condition_type"))
if (ckpt.get("config") or {}).get("condition_type") not in {"adagn", "film"}:
    raise RuntimeError("refusing non-AdaGN checkpoint")

vae = "/kaggle/working/outputs/vae_sr/checkpoints/latest.pt"
if not Path(vae).exists():
    vae = hf_hub_download(
        repo_id="HusseinHamouda/LatentSR-checkpoints",
        filename="vae_sr/latest.pt",
        local_dir="/kaggle/working/hf_ckpt",
    )

!python scripts/train_sr.py --config configs/latent_sr_adagn_q2.yaml --vae-checkpoint {vae} --resume {ckpt_path} --epochs 50 --device cuda --no-download

/kaggle/working/LatentSR


latent_sr_adagn_q2/latest.pt:   0%|          | 0.00/201M [00:00<?, ?B/s]

resume from: /kaggle/working/hf_ckpt/latent_sr_adagn_q2/latest.pt
epoch: 24
condition_type: adagn
condition_type=adagn  params=16.73M
Resumed from epoch 24, training to epoch 50
Device: cuda  |  AMP: True
Train pairs: 162770  |  batches/epoch: 2544  |  batch_size: 64
VAE: /kaggle/working/outputs/vae_sr/checkpoints/latest.pt  |  latent_scale: 1.0  |  condition: adagn
HF backup every epoch → hf://HusseinHamouda/LatentSR-checkpoints/latent_sr_adagn_q2/

Epoch [25/50]                                                                   
Train Loss:  0.136955
Time:        1596.4 sec
Val Loss:    0.136597
Saved /kaggle/working/outputs/latent_sr_adagn_q2/checkpoints/latest.pt (191.8 MB, epoch 25)
Saved snapshot checkpoint_epoch_025.pt
Generating qualitative SR grid (may take a few minutes)…
Saved comparison grid to /kaggle/working/outputs/latent_sr_adagn_q2/samples/sr_compare_epoch_025.png
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload           

In [6]:
%cd /kaggle/working/LatentSR

from pathlib import Path
from huggingface_hub import hf_hub_download
import torch

ckpt_path = hf_hub_download(
    repo_id="HusseinHamouda/LatentSR-checkpoints",
    filename="latent_sr_adagn_q2/latest.pt",
    local_dir="/kaggle/working/hf_ckpt",
)
print("resume from:", ckpt_path)

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print("epoch:", ckpt.get("epoch"))
print("condition_type:", (ckpt.get("config") or {}).get("condition_type"))
if (ckpt.get("config") or {}).get("condition_type") not in {"adagn", "film"}:
    raise RuntimeError("refusing non-AdaGN checkpoint")

vae = "/kaggle/working/outputs/vae_sr/checkpoints/latest.pt"
if not Path(vae).exists():
    vae = hf_hub_download(
        repo_id="HusseinHamouda/LatentSR-checkpoints",
        filename="vae_sr/latest.pt",
        local_dir="/kaggle/working/hf_ckpt",
    )

!python scripts/train_sr.py --config configs/latent_sr_adagn_q2.yaml --vae-checkpoint {vae} --resume {ckpt_path} --epochs 50 --device cuda --no-download

/kaggle/working/LatentSR


latent_sr_adagn_q2/latest.pt:   0%|          | 0.00/201M [00:00<?, ?B/s]

resume from: /kaggle/working/hf_ckpt/latent_sr_adagn_q2/latest.pt
epoch: 37
condition_type: adagn
condition_type=adagn  params=16.73M
Resumed from epoch 37, training to epoch 50
Device: cuda  |  AMP: True
Train pairs: 162770  |  batches/epoch: 2544  |  batch_size: 64
VAE: /kaggle/working/outputs/vae_sr/checkpoints/latest.pt  |  latent_scale: 1.0  |  condition: adagn
HF backup every epoch → hf://HusseinHamouda/LatentSR-checkpoints/latent_sr_adagn_q2/

Epoch [38/50]                                                                   
Train Loss:  0.135488
Time:        1594.5 sec
Saved /kaggle/working/outputs/latent_sr_adagn_q2/checkpoints/latest.pt (191.8 MB, epoch 38)
Saved snapshot checkpoint_epoch_038.pt
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ..._q2/checkpoints/latest.pt:   2%|▎             | 5.01MB /  201MB            

Processing Files (0 / 1)      :   2%|▎    

In [6]:
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="HusseinHamouda/LatentSR-checkpoints",
    filename="latent_sr_adagn_q2/latest.pt",
    local_dir="/content/hf_ckpt",
)
print(ckpt_path)

latent_sr_adagn_q2/latest.pt:   0%|          | 0.00/201M [00:00<?, ?B/s]

/content/hf_ckpt/latent_sr_adagn_q2/latest.pt


In [7]:
!hf download HusseinHamouda/LatentSR-checkpoints latent_sr_adagn_q2/latest.pt --local-dir /content/hf_ckpt


  A new version of huggingface_hub is available: 1.11.0 → 1.28.0

  Do you want to update now? [Y/n] (/usr/bin/python3 -m pip install -U huggingface_hub) ^C

✓ Downloaded
  path: /content/hf_ckpt/latent_sr_adagn_q2/latest.pt


In [12]:
%cd /kaggle/working/LatentSR
!pip install -q lpips

from pathlib import Path
from huggingface_hub import hf_hub_download

sr = Path("/kaggle/working/outputs/latent_sr_adagn_q2/checkpoints/latest.pt")
vae = Path("/kaggle/working/outputs/vae_sr/checkpoints/latest.pt")

if not sr.exists():
    sr = Path(hf_hub_download(
        repo_id="HusseinHamouda/LatentSR-checkpoints",
        filename="latent_sr_adagn_q2/latest.pt",
        local_dir="/kaggle/working/hf_ckpt",
    ))
if not vae.exists():
    vae = Path(hf_hub_download(
        repo_id="HusseinHamouda/LatentSR-checkpoints",
        filename="vae_sr/latest.pt",
        local_dir="/kaggle/working/hf_ckpt",
    ))

print("SR:", sr)
print("VAE:", vae)

!python scripts/evaluate.py --config configs/eval_sr.yaml --checkpoint {sr} --vae-checkpoint {vae} --output-dir /kaggle/working/outputs/eval_sr_adagn_q2_paired --seed 42 --num-images 64 --batch-size 4 --include-soft-decode --device cuda --no-download

/kaggle/working/LatentSR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.1 MB/s eta 0:00:00
SR: /kaggle/working/outputs/latent_sr_adagn_q2/checkpoints/latest.pt
VAE: /kaggle/working/outputs/vae_sr/checkpoints/latest.pt
SR epoch: 50
VAE: /kaggle/working/outputs/vae_sr/checkpoints/latest.pt
latent_scale: 1.0
data_dir: /kaggle/working/data/raw
Evaluating 64 val images on cuda …
Per-image noise seed=42 (x_T and every reverse step; independent of batch size; use the same seed for paired runs).
Note: each image runs a full reverse diffusion chain (slow on CPU).
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for

In [14]:
%cd /kaggle/working/LatentSR

!python scripts/compare_sr_evals.py --baseline /kaggle/working/outputs/eval_sr_q2_paired/per_image.csv --candidate /kaggle/working/outputs/eval_sr_adagn_q2_paired/per_image.csv --baseline-name concat_q2 --candidate-name adagn_q2 --output-dir /kaggle/working/outputs/eval_sr_adagn_vs_concat

/kaggle/working/LatentSR
Traceback (most recent call last):
  File "/kaggle/working/LatentSR/scripts/compare_sr_evals.py", line 89, in <module>
    main()
  File "/kaggle/working/LatentSR/scripts/compare_sr_evals.py", line 59, in main
    baseline = load_per_image_csv(args.baseline)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/LatentSR/src/latentsr/metrics/paired_stats.py", line 21, in load_per_image_csv
    with path.open(newline="", encoding="utf-8") as file:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 1013, in open
    return io.open(self, mode, buffering, encoding, errors, newline)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/outputs/eval_sr_q2_paired/per_image.csv'


In [7]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    print(p)

/kaggle/input/datasets
/kaggle/input/datasets/husseinhamouda
/kaggle/input/datasets/husseinhamouda/per-image-baseline
/kaggle/input/datasets/husseinhamouda/per-image
/kaggle/input/datasets/husseinhamouda/per-image-200
/kaggle/input/datasets/husseinhamouda/per-image-50
/kaggle/input/datasets/husseinhamouda/per-image-baseline/per_image_baseline.csv
/kaggle/input/datasets/husseinhamouda/per-image/per_image.csv
/kaggle/input/datasets/husseinhamouda/per-image-200/per_image_200.csv
/kaggle/input/datasets/husseinhamouda/per-image-50/per_image_50.csv


In [9]:
import shutil
from pathlib import Path

src = Path("/kaggle/input/datasets/husseinhamouda/per-image/per_image.csv")
dst = Path("/kaggle/working/outputs/eval_sr_q2_paired/per_image.csv")

dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)

print(f"Copied to: {dst}")

Copied to: /kaggle/working/outputs/eval_sr_q2_paired/per_image.csv


In [10]:
#baseline
import shutil
from pathlib import Path

src = Path("/kaggle/input/datasets/husseinhamouda/per-image-baseline/per_image_baseline.csv")
dst = Path("/kaggle/working/outputs/eval_guidance_n256_baseline/per_image.csv")

dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)

print(f"Copied to: {dst}")

Copied to: /kaggle/working/outputs/eval_guidance_n256_baseline/per_image.csv


In [11]:
#50
import shutil
from pathlib import Path

src = Path("/kaggle/input/datasets/husseinhamouda/per-image-50/per_image_50.csv")
dst = Path("/kaggle/working/outputs/eval_guidance_n256_late_l50/per_image.csv")

dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)

print(f"Copied to: {dst}")

Copied to: /kaggle/working/outputs/eval_guidance_n256_late_l50/per_image.csv


In [12]:
#200
import shutil
from pathlib import Path

src = Path("/kaggle/input/datasets/husseinhamouda/per-image-200/per_image_200.csv")
dst = Path("/kaggle/working/outputs/eval_guidance_n256_late_l200/per_image.csv")

dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)

print(f"Copied to: {dst}")

Copied to: /kaggle/working/outputs/eval_guidance_n256_late_l200/per_image.csv


In [13]:
print(Path("/kaggle/working/outputs/eval_sr_q2_paired/per_image.csv").exists())

True


In [20]:
%cd /kaggle/working/LatentSR

!python scripts/compare_sr_evals.py --baseline /kaggle/working/outputs/eval_sr_q2_paired/per_image.csv --candidate /kaggle/working/outputs/eval_sr_adagn_q2_paired/per_image.csv --baseline-name concat_q2 --candidate-name adagn_q2 --output-dir /kaggle/working/outputs/eval_sr_adagn_vs_concat

/kaggle/working/LatentSR
n=64  Δ = adagn_q2 − concat_q2
metric       role           Δ mean                 95% CI     p_perm   CI≠0
------------------------------------------------------------------------------
psnr         primary       -0.0317     [-0.0889, +0.0229]     0.2753  False
lpips        primary       +0.0011     [+0.0000, +0.0022]    0.05959   True
ssim         secondary     +0.0003     [-0.0009, +0.0014]     0.6315  False
edge_mae     secondary     +0.0006     [-0.0000, +0.0012]    0.07199  False

Spearman ρ (pre-registered):
  delta_psnr_soft_vs_sr: nan
  delta_lpips_soft_vs_sr: nan
  delta_psnr_sr_vs_bicubic: -0.1326
  delta_psnr_sr_vs_hr_edge: 0.1586

Wrote paired analysis under: /kaggle/working/outputs/eval_sr_adagn_vs_concat
  /kaggle/working/outputs/eval_sr_adagn_vs_concat/paired_stats.txt
  /kaggle/working/outputs/eval_sr_adagn_vs_concat/paired_stats.json
  /kaggle/working/outputs/eval_sr_adagn_vs_concat/paired_deltas.csv
  /kaggle/working/outputs/eval_sr_adagn_vs_c

In [15]:
%cd /kaggle/working/LatentSR
!git pull
!pip install -q -e . gdown huggingface_hub

from pathlib import Path
import torch
from huggingface_hub import hf_hub_download

Path("/kaggle/working/artifacts/vae").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/artifacts/latent_sr_q2").mkdir(parents=True, exist_ok=True)

!gdown "https://drive.google.com/uc?id=1fWFgoXywT1IGtELJoW1_Y0xZ0PGET7dM" -O /kaggle/working/artifacts/vae/checkpoint_epoch_050.pt
!gdown "https://drive.google.com/uc?id=1f7Gxz0wgoH4s4Au5CQWG5Ucka9anCwRa" -O /kaggle/working/artifacts/latent_sr_q2/latest.pt

vae1 = Path("/kaggle/working/artifacts/vae/checkpoint_epoch_050.pt")
vae_sr = Path("/kaggle/working/outputs/vae_sr/checkpoints/latest.pt")
q2_sr = Path("/kaggle/working/artifacts/latent_sr_q2/latest.pt")
vae1_sr = Path(hf_hub_download(
    repo_id="HusseinHamouda/LatentSR-checkpoints",
    filename="latest.pt",
    local_dir="/kaggle/working/hf_ckpt",
))

for name, path in {
    "VAE-1": vae1,
    "VAE-SR": vae_sr,
    "Phase-8 concat": vae1_sr,
    "Q2 concat": q2_sr,
}.items():
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    ctype = (ckpt.get("config") or {}).get("condition_type", "vae")
    print(f"{name:16} epoch={ckpt.get('epoch')}  condition={ctype}  {path}")

/kaggle/working/LatentSR
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for latentsr (pyproject.toml) ... done
Downloading...
From (original): https://drive.google.com/uc?id=1fWFgoXywT1IGtELJoW1_Y0xZ0PGET7dM
From (redirected): https://drive.google.com/uc?id=1fWFgoXywT1IGtELJoW1_Y0xZ0PGET7dM&confirm=t&uuid=6cbb5bf9-573b-46dc-98cd-b0ddc976c3cf
To: /kaggle/working/artifacts/vae/checkpoint_epoch_050.pt
100%|██████████████████████████████████████| 83.6M/83.6M [00:01<00:00, 54.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1f7Gxz0wgoH4s4Au5CQWG5Ucka9anCwRa
From (redirected): https://drive.google.com/uc?id=1f7Gxz0wgoH4s4Au5CQWG5Ucka9anCwRa&confirm=t&uuid=de8fab24-b607-4c51-b3ba-947df717889c
To: /kaggle/working/artifacts/latent_sr_q2/latest.pt
100%|█████████████████████████

latest.pt:   0%|          | 0.00/198M [00:00<?, ?B/s]

VAE-1            epoch=50  condition=vae  /kaggle/working/artifacts/vae/checkpoint_epoch_050.pt
VAE-SR           epoch=20  condition=vae  /kaggle/working/outputs/vae_sr/checkpoints/latest.pt
Phase-8 concat   epoch=50  condition=vae  /kaggle/working/hf_ckpt/latest.pt
Q2 concat        epoch=50  condition=vae  /kaggle/working/artifacts/latent_sr_q2/latest.pt


In [7]:
!python scripts/diagnose_timesteps.py --config configs/eval_sr.yaml --baseline-sr {vae1_sr} --baseline-vae {vae1} --candidate-sr {q2_sr} --candidate-vae {vae_sr} --output-dir /kaggle/working/outputs/eval_timestep_diagnostic --num-images 64 --batch-size 4 --seed 42 --device cuda --no-download

baseline (vae1): epoch=50 condition=concat
candidate (vae_sr): epoch=50 condition=concat
Diagnosing 64 val images, 1000 steps, seed=42 on cuda
                                                                                
n=64  t=0 (clean) … t=999 (noise)
Δz_lr = ||z_lr[vae_sr] − z_lr[vae1]||  (RMSE)

    t     Δz_lr       cos ẑ0,z_lr            ||ẑ0−z_lr||        ||ẑ0SR−ẑ0VAE1||
                      VAE-1     VAE-SR       VAE-1     VAE-SR
----------------------------------------------------------------------------------------
    0    0.5202      0.6152     0.7862      0.7753     0.5716            0.2394
  500    0.5202      0.7709     0.9688      0.5310     0.1828            0.2066
  999    0.5202      0.2819     0.5427      2.5404     1.5241            2.4541

Wrote /kaggle/working/outputs/eval_timestep_diagnostic
  /kaggle/working/outputs/eval_timestep_diagnostic/timestep_means.csv
  /kaggle/working/outputs/eval_timestep_diagnostic/timestep_diagnostic.json
  /kaggle/working/outp

In [9]:
%cd /kaggle/working/LatentSR
!git pull
!pip install -q -e . lpips

!python scripts/diagnose_z0_recon.py --config configs/eval_sr.yaml --baseline-sr {vae1_sr} --baseline-vae {vae1} --candidate-sr {q2_sr} --candidate-vae {vae_sr} --output-dir /kaggle/working/outputs/eval_z0_recon --num-images 64 --batch-size 4 --seed 42 --device cuda --no-download

/kaggle/working/LatentSR
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.0 MB/s eta 0:00:00
  Building editable for latentsr (pyproject.toml) ... done
baseline (vae1): epoch=50 condition=concat
candidate (vae_sr): epoch=50 condition=concat
decode ẑ0 at t=[800, 700, 650, 600, 500, 400, 300, 200, 100, 0]
Diagnosing 64 val images, 1000 reverse steps, seed=42 on cuda
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other 

# ---------------------------------------------------

In [14]:
%cd /kaggle/working/LatentSR
!git pull

/kaggle/working/LatentSR
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 597 bytes | 298.00 KiB/s, done.
From https://github.com/HusseinHanafy207/LatentSR
   88bcb13..6402b54  main       -> origin/main
Updating 88bcb13..6402b54
Fast-forward
 scripts/evaluate_guidance.py | 4 ++--
 scripts/guidance_sanity.py   | 4 ++--
 2 files changed, 4 insertions(+), 4 deletions(-)


In [15]:
!python scripts/guidance_sanity.py \
  --config configs/eval_sr.yaml \
  --sr-checkpoint /kaggle/working/artifacts/latent_sr_q2/latest.pt \
  --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt \
  --output-dir /kaggle/working/outputs/eval_guidance_sanity \
  --device cuda --no-download

Reference (n=64, seed=42): bicubic 26.13 dB / LPIPS 0.282  |  soft-decode 28.48 / 0.119  |  unguided LatentSR 26.48 / 0.0685

Pre-registered interpretation (fix BEFORE looking at Stage-1 numbers)
---------------------------------------------------------------------
PSNR↑, LPIPS ≈0.068 or better
    Strong positive: conditioning preservation helps without perceptual collapse.
PSNR↑ but LPIPS moving toward 0.119
    Over-guidance / soft-decode collapse.
PSNR ≈ baseline, LPIPS ≈ baseline, but cos(ẑ0, z_lr) clearly higher
    Mechanistic null: conditioning preserved, does not improve the final sample.
PSNR ≈ baseline, LPIPS ≈ baseline, AND trajectory barely differs
    Implementation/scale failure — re-check λ_g; do not report as a scientific null.
Both early and late show a result
    Stage 2 strength sweep on the better window.
Only early or only late shows a result
    Stop. Localize the conditioning-loss region. Do not run a combined schedule.
Neither works, but trajectory confirms gui

In [14]:
%cd /kaggle/working/LatentSR
!git pull
!pip install -q -e . lpips

/kaggle/working/LatentSR
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.2 MB/s eta 0:00:00
  Building editable for latentsr (pyproject.toml) ... done


In [18]:
!python scripts/evaluate_guidance.py --condition baseline --lambda-g 0 \
  --config configs/eval_sr.yaml \
  --sr-checkpoint /kaggle/working/artifacts/latent_sr_q2/latest.pt \
  --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt \
  --output-dir /kaggle/working/outputs/eval_guidance_baseline \
  --seed 42 --num-images 64 --device cuda --no-download

!python scripts/evaluate_guidance.py --condition early --lambda-g 200 \
  --config configs/eval_sr.yaml \
  --sr-checkpoint /kaggle/working/artifacts/latent_sr_q2/latest.pt \
  --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt \
  --compare-baseline /kaggle/working/outputs/eval_guidance_baseline/per_image.csv \
  --output-dir /kaggle/working/outputs/eval_guidance_early \
  --seed 42 --num-images 64 --batch-size 1 --device cuda --no-download

!python scripts/evaluate_guidance.py --condition late --lambda-g 200 \
  --config configs/eval_sr.yaml \
  --sr-checkpoint /kaggle/working/artifacts/latent_sr_q2/latest.pt \
  --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt \
  --compare-baseline /kaggle/working/outputs/eval_guidance_baseline/per_image.csv \
  --output-dir /kaggle/working/outputs/eval_guidance_late \
  --seed 42 --num-images 64 --batch-size 1 --device cuda --no-download

Reference (n=64, seed=42): bicubic 26.13 dB / LPIPS 0.282  |  soft-decode 28.48 / 0.119  |  unguided LatentSR 26.48 / 0.0685

Pre-registered interpretation (fix BEFORE looking at Stage-1 numbers)
---------------------------------------------------------------------
PSNR↑, LPIPS ≈0.068 or better
    Strong positive: conditioning preservation helps without perceptual collapse.
PSNR↑ but LPIPS moving toward 0.119
    Over-guidance / soft-decode collapse.
PSNR ≈ baseline, LPIPS ≈ baseline, but cos(ẑ0, z_lr) clearly higher
    Mechanistic null: conditioning preserved, does not improve the final sample.
PSNR ≈ baseline, LPIPS ≈ baseline, AND trajectory barely differs
    Implementation/scale failure — re-check λ_g; do not report as a scientific null.
Both early and late show a result
    Stage 2 strength sweep on the better window.
Only early or only late shows a result
    Stop. Localize the conditioning-loss region. Do not run a combined schedule.
Neither works, but trajectory confirms gui

In [19]:
!python scripts/evaluate_guidance.py --condition late --lambda-g 50 \
  --config configs/eval_sr.yaml \
  --sr-checkpoint /kaggle/working/artifacts/latent_sr_q2/latest.pt \
  --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt \
  --compare-baseline /kaggle/working/outputs/eval_guidance_baseline/per_image.csv \
  --output-dir /kaggle/working/outputs/eval_guidance_late_l50 \
  --seed 42 --num-images 64 --batch-size 1 --device cuda --no-download

!python scripts/evaluate_guidance.py --condition late --lambda-g 800 \
  --config configs/eval_sr.yaml \
  --sr-checkpoint /kaggle/working/artifacts/latent_sr_q2/latest.pt \
  --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt \
  --compare-baseline /kaggle/working/outputs/eval_guidance_baseline/per_image.csv \
  --output-dir /kaggle/working/outputs/eval_guidance_late_l800 \
  --seed 42 --num-images 64 --batch-size 1 --device cuda --no-download

Reference (n=64, seed=42): bicubic 26.13 dB / LPIPS 0.282  |  soft-decode 28.48 / 0.119  |  unguided LatentSR 26.48 / 0.0685

Pre-registered interpretation (fix BEFORE looking at Stage-1 numbers)
---------------------------------------------------------------------
PSNR↑, LPIPS ≈0.068 or better
    Strong positive: conditioning preservation helps without perceptual collapse.
PSNR↑ but LPIPS moving toward 0.119
    Over-guidance / soft-decode collapse.
PSNR ≈ baseline, LPIPS ≈ baseline, but cos(ẑ0, z_lr) clearly higher
    Mechanistic null: conditioning preserved, does not improve the final sample.
PSNR ≈ baseline, LPIPS ≈ baseline, AND trajectory barely differs
    Implementation/scale failure — re-check λ_g; do not report as a scientific null.
Both early and late show a result
    Stage 2 strength sweep on the better window.
Only early or only late shows a result
    Stop. Localize the conditioning-loss region. Do not run a combined schedule.
Neither works, but trajectory confirms gui

### ------------------------------

In [12]:
!python scripts/run_guidance_n256.py --job baseline \
  --sr-checkpoint /kaggle/working/artifacts/latent_sr_q2/latest.pt \
  --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt \
  --output-root /kaggle/working/outputs --device cuda --no-download

Reference (n=64, seed=42): bicubic 26.13 dB / LPIPS 0.282  |  soft-decode 28.48 / 0.119  |  unguided LatentSR 26.48 / 0.0685

Pre-registered interpretation (fix BEFORE looking at Stage-1 numbers)
---------------------------------------------------------------------
PSNR↑, LPIPS ≈0.068 or better
    Strong positive: conditioning preservation helps without perceptual collapse.
PSNR↑ but LPIPS moving toward 0.119
    Over-guidance / soft-decode collapse.
PSNR ≈ baseline, LPIPS ≈ baseline, but cos(ẑ0, z_lr) clearly higher
    Mechanistic null: conditioning preserved, does not improve the final sample.
PSNR ≈ baseline, LPIPS ≈ baseline, AND trajectory barely differs
    Implementation/scale failure — re-check λ_g; do not report as a scientific null.
Both early and late show a result
    Stage 2 strength sweep on the better window.
Only early or only late shows a result
    Stop. Localize the conditioning-loss region. Do not run a combined schedule.
Neither works, but trajectory confirms gui

In [13]:
!python scripts/run_guidance_n256.py --job late50 \
  --sr-checkpoint /kaggle/working/artifacts/latent_sr_q2/latest.pt \
  --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt \
  --output-root /kaggle/working/outputs --device cuda --no-download

Reference (n=64, seed=42): bicubic 26.13 dB / LPIPS 0.282  |  soft-decode 28.48 / 0.119  |  unguided LatentSR 26.48 / 0.0685

Pre-registered interpretation (fix BEFORE looking at Stage-1 numbers)
---------------------------------------------------------------------
PSNR↑, LPIPS ≈0.068 or better
    Strong positive: conditioning preservation helps without perceptual collapse.
PSNR↑ but LPIPS moving toward 0.119
    Over-guidance / soft-decode collapse.
PSNR ≈ baseline, LPIPS ≈ baseline, but cos(ẑ0, z_lr) clearly higher
    Mechanistic null: conditioning preserved, does not improve the final sample.
PSNR ≈ baseline, LPIPS ≈ baseline, AND trajectory barely differs
    Implementation/scale failure — re-check λ_g; do not report as a scientific null.
Both early and late show a result
    Stage 2 strength sweep on the better window.
Only early or only late shows a result
    Stop. Localize the conditioning-loss region. Do not run a combined schedule.
Neither works, but trajectory confirms gui

In [ ]:
!python scripts/run_guidance_n256.py --job late200 \
  --sr-checkpoint /kaggle/working/artifacts/latent_sr_q2/latest.pt \
  --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt \
  --output-root /kaggle/working/outputs --device cuda --no-download

Reference (n=64, seed=42): bicubic 26.13 dB / LPIPS 0.282  |  soft-decode 28.48 / 0.119  |  unguided LatentSR 26.48 / 0.0685

Pre-registered interpretation (fix BEFORE looking at Stage-1 numbers)
---------------------------------------------------------------------
PSNR↑, LPIPS ≈0.068 or better
    Strong positive: conditioning preservation helps without perceptual collapse.
PSNR↑ but LPIPS moving toward 0.119
    Over-guidance / soft-decode collapse.
PSNR ≈ baseline, LPIPS ≈ baseline, but cos(ẑ0, z_lr) clearly higher
    Mechanistic null: conditioning preserved, does not improve the final sample.
PSNR ≈ baseline, LPIPS ≈ baseline, AND trajectory barely differs
    Implementation/scale failure — re-check λ_g; do not report as a scientific null.
Both early and late show a result
    Stage 2 strength sweep on the better window.
Only early or only late shows a result
    Stop. Localize the conditioning-loss region. Do not run a combined schedule.
Neither works, but trajectory confirms gui

In [16]:
!python scripts/run_guidance_n256.py --job late800 \
  --sr-checkpoint /kaggle/working/artifacts/latent_sr_q2/latest.pt \
  --vae-checkpoint /kaggle/working/outputs/vae_sr/checkpoints/latest.pt \
  --output-root /kaggle/working/outputs --device cuda --no-download

Reference (n=64, seed=42): bicubic 26.13 dB / LPIPS 0.282  |  soft-decode 28.48 / 0.119  |  unguided LatentSR 26.48 / 0.0685

Pre-registered interpretation (fix BEFORE looking at Stage-1 numbers)
---------------------------------------------------------------------
PSNR↑, LPIPS ≈0.068 or better
    Strong positive: conditioning preservation helps without perceptual collapse.
PSNR↑ but LPIPS moving toward 0.119
    Over-guidance / soft-decode collapse.
PSNR ≈ baseline, LPIPS ≈ baseline, but cos(ẑ0, z_lr) clearly higher
    Mechanistic null: conditioning preserved, does not improve the final sample.
PSNR ≈ baseline, LPIPS ≈ baseline, AND trajectory barely differs
    Implementation/scale failure — re-check λ_g; do not report as a scientific null.
Both early and late show a result
    Stage 2 strength sweep on the better window.
Only early or only late shows a result
    Stop. Localize the conditioning-loss region. Do not run a combined schedule.
Neither works, but trajectory confirms gui

In [17]:
!python scripts/run_guidance_n256.py --job compare --output-root /kaggle/working/outputs

Paired comparison: /kaggle/working/outputs/eval_guidance_n256_late_l50_vs_baseline
Paired comparison: /kaggle/working/outputs/eval_guidance_n256_late_l200_vs_baseline
Paired comparison: /kaggle/working/outputs/eval_guidance_n256_late_l800_vs_baseline


# ---------------------------------------------------
## Representation geometry (RiT-style): VAE-1 vs VAE-SR

No diffusion. Encodes the same CelebA val images under frozen **VAE-1** and **VAE-SR**, then scores:

- TwoNN intrinsic dimensionality
- Effective rank
- Covariance / transport condition number kappa(Sigma_t)
- Excess kurtosis
- PCA cumulative variance spectrum

Reports both `z_hr = encode(HR)` and `z_lr = encode(bicubic↑ LR)`.

Needs the geometry script on `main` (`scripts/diagnose_representation_geometry.py`). Prefer **N ≥ 2048** (ambient D = 4096). Smoke-test with `NUM_IMAGES = 256` first.

**Note:** the output folder stays empty until the end (except `status.json`). The old hang was loading the unused CelebA **train** split + redundant TwoNN passes — fixed; re-`git pull` before running.


In [ ]:
%cd /kaggle/working/LatentSR
!git pull
!pip install -q -e . gdown huggingface_hub


In [ ]:
from pathlib import Path
import shutil
import torch
from huggingface_hub import hf_hub_download

REPO = "HusseinHamouda/LatentSR-checkpoints"
HF_DIR = "/kaggle/working/hf_ckpt"

Path("/kaggle/working/artifacts/vae").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/outputs/vae_sr/checkpoints").mkdir(parents=True, exist_ok=True)

# VAE-1 (recon epoch 50) — same Drive ID as the timestep-diagnostic cell; HF fallback.
vae1 = Path("/kaggle/working/artifacts/vae/checkpoint_epoch_050.pt")
if not vae1.is_file():
    !gdown "https://drive.google.com/uc?id=1fWFgoXywT1IGtELJoW1_Y0xZ0PGET7dM" -O /kaggle/working/artifacts/vae/checkpoint_epoch_050.pt
if not vae1.is_file():
    src = Path(hf_hub_download(
        repo_id=REPO,
        filename="vae/checkpoint_epoch_050.pt",
        local_dir=HF_DIR,
    ))
    shutil.copy2(src, vae1)

# VAE-SR (align epoch 20)
vae_sr = Path("/kaggle/working/outputs/vae_sr/checkpoints/latest.pt")
if not vae_sr.is_file():
    src = Path(hf_hub_download(
        repo_id=REPO,
        filename="vae_sr/latest.pt",
        local_dir=HF_DIR,
    ))
    if src.resolve() != vae_sr.resolve():
        shutil.copy2(src, vae_sr)

for name, path in {"VAE-1": vae1, "VAE-SR": vae_sr}.items():
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print(f"{name:8} epoch={ckpt.get('epoch')}  keys_ok={'model_state_dict' in ckpt}  {path}")
    if "model_state_dict" not in ckpt:
        raise RuntimeError(f"{name} checkpoint missing model_state_dict")


In [ ]:
%cd /kaggle/working/LatentSR
!git pull
!pip install -q -e .

from pathlib import Path

NUM_IMAGES = 2048  # 256 = smoke test; 2048+ recommended for D=4096
BATCH_SIZE = 32

vae1 = Path("/kaggle/working/artifacts/vae/checkpoint_epoch_050.pt")
vae_sr = Path("/kaggle/working/outputs/vae_sr/checkpoints/latest.pt")
out = Path("/kaggle/working/outputs/eval_representation_geometry")
out.mkdir(parents=True, exist_ok=True)

print("VAE-1:", vae1)
print("VAE-SR:", vae_sr)
print("N:", NUM_IMAGES, "batch:", BATCH_SIZE)
print("Watch status.json while it runs — metrics land only at the end.")

!python scripts/diagnose_representation_geometry.py \
  --baseline-vae {vae1} \
  --candidate-vae {vae_sr} \
  --baseline-name vae1 \
  --candidate-name vae_sr \
  --config configs/eval_vae.yaml \
  --data-dir /kaggle/working/data/raw \
  --output-dir {out} \
  --num-images {NUM_IMAGES} \
  --batch-size {BATCH_SIZE} \
  --num-workers 0 \
  --seed 42 \
  --twonn-bootstraps 10 \
  --device cuda \
  --no-download

print("\nstatus:", (out / "status.json").read_text() if (out / "status.json").exists() else "missing")
print("Outputs:")
for p in sorted(out.iterdir()):
    print(" ", p.name)
